# Hinode/EIS データ解析講習会

**Solar-C (EUVST) の準備**として、いま手に入る Hinode/EIS のデータで
分光解析をひととおり通します。

EUVST の EUV バンド 170–215 Å は EIS の短波長帯とほぼ同じで、**同じ輝線を撮ります**。
**今日やることは、そのまま 2028 年に使えます。**

| 章 | 内容 | 目安 |
|---|---|---|
| 1 | EIS のデータを見る | 20 分 |
| 2 | フィットして**強度**を出す | 35 分 |
| 3 | **速度**を出す | 50 分 |
| 4 | 線幅から**非熱的速度**を出す | 30 分 |
| 5 | **温度分布 (DEM)** を出す | 30 分 |
| 付録 | 自分の研究で使うときに読む（当日は走らせません） | — |

上から順に実行してください。インストールとデータ取得は**最初の 1 回だけ**です。


<!-- 準備は 00_setup 側に書いてある -->

In [ ]:
!pip install -q eispac fiasco demregpy

### インストール直後のランタイム再起動について
#
Colab では、pip が `numpy` などを入れ替えると、**実行中のセッションが
古いモジュールを掴んだまま**になり、あとで次のようなエラーが出ることがある:
#
```
ImportError: cannot import name '_center' from 'numpy._core.umath'
```
#
これはインストールの失敗ではなく、**再起動すれば直る**。
次のセルが入れ替えを検出して、必要なときだけ自動で再起動する。
#
**再起動が起きたら、もう一度このノートを先頭から実行すること。**
2 回目はインストールもダウンロードも済んでいるので一瞬で終わる。

In [ ]:
import sys
from importlib.metadata import version

need_restart = False
try:
    loaded = sys.modules["numpy"].__version__ if "numpy" in sys.modules else None
    if loaded is not None and loaded != version("numpy"):
        need_restart = True
        print(f"numpy が {loaded} -> {version('numpy')} に入れ替わりました")
except Exception as e:                      # 判定自体が失敗したら念のため再起動
    need_restart = True
    print("numpy の状態を確認できませんでした:", e)

if need_restart:
    print("ランタイムを再起動します。"
          "再起動したら、もう一度このノートを先頭から実行してください。")
    try:
        import IPython
        ipy = IPython.get_ipython()
        if ipy is not None:
            ipy.kernel.do_shutdown(True)    # Colab のランタイム再起動
    except Exception:
        import os
        os.kill(os.getpid(), 9)
else:
    print("numpy の入れ替えは起きていません。このまま先へ進んで大丈夫です。")

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/hottahd/EIS_practice.git"
if not os.path.exists("scripts/lines_warren2012.py"):      # リポジトリの外にいる
    if not os.path.exists("EIS_practice"):
        print("教材リポジトリを取得中 ...")
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("EIS_practice")
sys.path.insert(0, "scripts")
print("作業ディレクトリ:", os.getcwd())

# Hinode/EIS データ解析講習会

## この講習会について

**狙いは Solar-C (EUVST) の準備**です。打ち上げたその日から解析できるように、
いま手に入るデータで実力をつけておきます。

EUVST は EIS と同じ**スリット走査型の EUV 分光器**で、
**EUV バンド 170–215 Å は EIS の短波長帯 171–212 Å とほぼ同じ**です。
つまり**同じ輝線を撮ります**。今日フィットする Fe XII や Ca XV は、
そのまま EUVST の主力になります。

| | EUVST (2028 打ち上げ予定) | Hinode/EIS (2006–) |
|---|---|---|
| 波長帯 | **170–215 Å** + 460–1220 Å | 171–212 / 245–291 Å |
| 温度被覆 | 2×10⁴ – 1.5×10⁷ K（シームレス） | 飛び飛び |
| 空間分解能 | **0.4″** | ~2″ |
| カデンス | **1 秒** | 数十秒 |
| 実効面積 | **10–30 倍** | — |

**今日やることは、そのまま 2028 年に使えます。**

## 今日の内容

| 章 | 内容 |
|---|---|
| 1 | EIS のデータを見る |
| 2 | フィットして**強度**を出す |
| 3 | **速度**を出す |
| 4 | 線幅から**非熱的速度**を出す |
| 5 | **温度分布 (DEM)** を出す |

題材は **2011 年 7 月 2 日 03:07 UT、活動領域 NOAA 1243**。
Warren, Winebarger & Brooks (2012), ApJ 759, 141 が使ったデータで、
**論文に観測値の表が載っている**ので、自分の結果と答え合わせができます。

## 準備

パッケージを入れて、教材リポジトリとデータを取ってきます。
**観測データは必要になったところで自動的に取得**されます（既にあれば何もしません）。

### 赤い `ERROR:` が出ても、たいていは無視してよい

Colab では `google-colab 1.0.0 requires requests==2.32.4, but you have ...`
のような行が出ることがあります。**インストールの失敗ではなく**、
Colab に元から入っている別のパッケージとの食い違いの報告です。
教材で使うものは正しく入っています。

次のセルは、`numpy` が入れ替わった場合だけランタイムを再起動します
（再起動したら、もう一度先頭から実行してください。2 回目は一瞬で終わります）。

In [ ]:
import eispac
import sunpy
import numpy as np

from workshop import ensure_eis

print("eispac", eispac.__version__, " sunpy", sunpy.__version__,
      " numpy", np.__version__)
ensure_eis()          # EIS の level-1 データ（94 MB）。既にあれば何もしない
print("準備完了")

**★ Colab の保存について**

GitHub から開いたノートは**読み取り専用の一時セッション**です。

- 編集や実行結果を残したいときは「ファイル → ドライブにコピーを保存」
- 仮想マシンが切れるとダウンロードしたデータも消えますが、
  **上から流し直せば復帰します**（既にあるファイルは取得し直しません）

# 第 1 章: EIS のデータを見る

**20 分**

EIS が何を撮っているのかを、実物で確かめます。

## 1-1. EIS はスペクトルを撮る。画像は自分で作る

EIS は細長いスリット（1″ × 512″）を太陽に当て、その 1 次元の像を
波長分散させて CCD に落とします。**1 回の露出で得られるのは
(空間 512) × (波長) の 2 次元**で、画像ではありません。

2 次元の画像がほしければ**スリットを横に振ります**。これが**ラスター**です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import eispac

from workshop import EIS_FILE, ensure_eis

ensure_eis()
cube = eispac.read_cube(EIS_FILE, 195.119)      # Fe XII 195.119 Å
print("shape (y, x, wavelength) =", cube.data.shape)
print("単位 =", cube.unit)

**`(512, 60, 24)`**

| 軸 | 数 | 正体 |
|---|---|---|
| 0 | 512 | **スリットに沿った空間**（1″/画素） |
| 1 | 60 | **ラスターのステップ**（2″/画素） |
| 2 | 24 | **波長**（0.0223 Å/画素） |

`read_cube` は、指定した波長を含む**スペクトルウィンドウ**を丸ごと読みます。

## 1-2. スペクトルを 1 本見る

In [ ]:
y, x = 250, 35          # 活動領域コアの中の 1 画素
plt.figure(figsize=(7, 4))
plt.plot(cube.wavelength[y, x, :], cube.data[y, x, :], "o-", ms=4)
plt.axvline(195.119, color="r", ls="--", lw=1, label="Fe XII 195.119")
plt.xlabel("wavelength [Å]")
plt.ylabel(f"intensity [{cube.unit}]")
plt.title(f"EIS spectrum, single pixel (y={y}, x={x})")
plt.legend()
plt.tight_layout()
plt.show()

- 山が 1 つ。これが Fe XII 195.119 Å。**幅は 3–4 画素**
- 台が浮いている。これが背景（連続光・散乱光）
- **この山の面積が輝線強度**、**中心のずれが速度**、**幅が乱れ** —— 第 2〜4 章でやります

## 1-3. 輝線を変えると、まったく別の太陽が見える

波長方向に積めば強度マップになります（フィットせずに数秒でできる簡易版）。
**形成温度の順**に 8 枚並べます。

In [ ]:
def raster_image(datafile, wvl):
    """波長方向に積んで強度マップにする。"""
    c = eispac.read_cube(datafile, wvl)
    # 欠損サンプルは平均に入れない（理由は付録 A）
    d = np.where(np.asarray(c.mask, dtype=bool), np.nan, c.data)
    return np.nanmean(d, axis=2) * d.shape[2]


PANELS = [
    (275.368, "Si VII 275.4",  "0.6 MK  moss"),
    (184.536, "Fe X 184.5",    "1.1 MK"),
    (195.119, "Fe XII 195.1",  "1.6 MK"),
    (202.044, "Fe XIII 202.0", "1.8 MK"),
    (262.984, "Fe XVI 263.0",  "2.8 MK"),
    (193.874, "Ca XIV 193.9",  "3.5 MK"),
    (200.972, "Ca XV 201.0",   "4.5 MK"),
    (192.858, "Ca XVII 192.9", "5.6 MK"),
]

ext = cube.meta["extent_arcsec"]        # [x0, x1, y0, y1]（太陽面座標, arcsec）
fig, axes = plt.subplots(1, len(PANELS), figsize=(2.3 * len(PANELS), 9))
for ax, (wvl, label, temp) in zip(axes, PANELS):
    v = np.sqrt(np.clip(raster_image(EIS_FILE, wvl), 0, None))   # sqrt で暗部を持ち上げる
    lo, hi = np.nanpercentile(v, [1, 99.5])
    ax.imshow(v, origin="lower", extent=ext, aspect="equal",
              cmap="inferno", vmin=lo, vmax=hi)
    ax.set_title(f"{label}\n{temp}", fontsize=9)
    ax.set_xlabel("Solar X [″]")
    if ax is not axes[0]:
        ax.set_yticklabels([])
axes[0].set_ylabel("Solar Y [″]")
fig.suptitle("NOAA 1243   2011-07-02 03:07 UT   (same field of view)", fontsize=12)
fig.tight_layout(rect=[0, 0.01, 1, 0.965])
plt.show()

**同じ場所なのに、別物に見えます。**

| 温度 | 見えるもの |
|---|---|
| Si VII (0.6 MK) | **まだら模様** = moss。高温ループの**足元**が遷移層で光っている |
| Fe X–XIII (1–2 MK) | **細いループ**が何本も。周辺まで広がる |
| Fe XVI 以上 (2.8 MK–) | ループが消え、**中心部の塊**だけ = 活動領域コア |

コロナが単一温度なら、どの輝線でも同じ絵になるはずです。
そうならないのは、**視線上に色々な温度のプラズマが混ざっている**から。
その量を温度ごとに測るのが **DEM 解析**（第 5 章）です。

**図のラベルが英語なのは**、Colab に日本語フォントが無いためです
（日本語だと豆腐になります）。

## 1-4. この「画像」は同時刻ではない

スリットを 1 ステップずつ動かすので、**x 軸は空間であると同時に時間軸**です。

In [ ]:
from astropy.time import Time

h = cube.meta["index"]
t0, t1 = Time(h["date_obs"]), Time(h["date_end"])
total = (t1 - t0).to_value("s")

print("観測プログラム :", h["stud_acr"], " (提案者:", h["st_auth"] + ")")
print("開始 / 終了    :", h["date_obs"], "/", h["date_end"])
print(f"所要時間       : {total/60:.1f} 分  ({h['nraster']} ステップ)")
print(f"1 ステップ     : {total/h['nraster']:.0f} 秒")
print(f"視野           : {h['fovx']:.0f}″ x {h['fovy']:.0f}″"
      f"  （x は {h['fovx']/h['nraster']:.1f}″/step、スリット幅は {h['slit_id']}）")

**62 分かかっています。** 左端と右端では 1 時間離れている。

- 時間変化する現象（フレア、ジェット）には使えない
- 「速度マップ」も同時刻の速度場ではない
- 一方、定常的な構造を測るなら問題なく、むしろ S/N の面で有利

**Solar-C EUVST はカデンス 1 秒**なので、この制約は大きく緩みます。
ただし「ラスターは掃いて作る」こと自体は変わりません。

## 1-5. 演習

1. `PANELS` に自分で輝線を足して描く。使える波長は下の一覧から選ぶ
2. 1-2 のスペクトルを、**moss の上**（`y=250, x=10` 付近）と
   **コアの中**（`y=250, x=35`）で描き比べる。Si VII 275.368 でやると差が大きい

In [ ]:
wi = eispac.read_wininfo(EIS_FILE.replace(".data.h5", ".head.h5"))
print(f"この観測に入っているスペクトルウィンドウ（{len(wi)} 個）")
print(f"{'#':>3} {'line_id':<22} {'wvl_min':>9} {'wvl_max':>9}")
for w in wi:
    print(f"{w['iwin']:3d} {str(w['line_id']):<22} {w['wvl_min']:9.3f} {w['wvl_max']:9.3f}")

**どの輝線が使えるかは、観測プログラムの設計時に決まっています。**
全波長を降ろすとテレメトリが足りないので、必要な輝線の周りだけを切り出します。
この観測は Ca XIV–XVII を含んでいるので、3 MK 以上を測れます。

# 第 2 章: フィットして強度を出す

**35 分**

山にガウシアンを当てて、**輝線強度という数字**を取り出します。

## 2-1. 何を測るのか

$$ I(\lambda) = A\exp\left[-\frac{(\lambda-\lambda_0)^2}{2\sigma^2}\right] + b $$

フィットで 3 つの量が出ます。

| パラメータ | 物理量 | 使う章 |
|---|---|---|
| 面積 $A\sigma\sqrt{2\pi}$ | **輝線強度** [erg cm⁻² s⁻¹ sr⁻¹] | 第 2・5 章 |
| 中心 $\lambda_0$ | **ドップラー速度** | 第 3 章 |
| 幅 $\sigma$ | **熱運動 + 非熱的速度** | 第 4 章 |

**強度は測定値ではなく、フィットの産物**です。背景をどこに引くか、
隣の線をどう扱うか、ガウシアンを何本当てるか —— すべてモデルの仮定です。

## 2-2. eispac のテンプレート

輝線ごとのフィット設定（**テンプレート**）が同梱されています。

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import eispac

from workshop import EIS_FILE, ensure_eis

ensure_eis()
path = eispac.data.get_fit_template_filepath("fe_12_195_119.2c.template.h5")
tmplt = eispac.read_template(path)

print("ファイル :", os.path.basename(path))
print("line_ids :", tmplt.template["line_ids"])
print("ガウシアンの本数 :", tmplt.template["n_gauss"])
print()
print(f"{'#':>2} {'初期値':>12} {'下限〜上限':>24} {'tied（他に縛る）':>18}")
for i, p in enumerate(tmplt.parinfo):
    lim = f"{p['limits'][0]:.3f} 〜 {p['limits'][1]:.3f}" if p["limited"].any() else "—"
    print(f"{i:2d} {p['value']:12.4f} {lim:>24} {str(p['tied']):>18}")

パラメータは **[振幅, 中心, 幅] × ガウシアンの本数 + 背景**。
`2c` は 2 成分なので 3×2 + 1 = 7 個です。

Fe XII 195.119 に 2 成分あるのは、195.179 Å に弱い Fe XII があるためです
（密度が高いと無視できない）。第 2 成分は `tied` で
**位置と幅を第 1 成分に縛って**います（原子データで決め打ち）。

### ★ `line_ids` を必ず確認してから使う

**目的の線が第 0 成分とは限りません。**

In [ ]:
for name in ["fe_12_195_119.2c", "fe_13_203_826.2c", "fe_14_270_519.2c",
             "ar_14_194_396.2c", "ca_14_193_874.2c"]:
    t = eispac.read_template(eispac.data.get_fit_template_filepath(name + ".template.h5"))
    print(f"{name:<20} {[str(s) for s in t.template['line_ids']]}")

`fe_13_203_826.2c` の**第 0 成分は Fe XII 203.720** です。
`component=0` と書くと、Fe XIII のつもりで**別のイオンの強度**を取ってしまいます。
エラーは出ません。**波長で照合して成分番号を決める**のが安全です。

In [ ]:
def pick_component(template, target_wvl):
    """line_ids を見て、目的波長に一番近い成分の番号を返す。"""
    ids = [str(s) for s in template.template["line_ids"]]
    best, bestd = 0, 1e9
    for i, s in enumerate(ids):
        try:
            w = float(s.split()[-1])
        except ValueError:
            continue
        if abs(w - target_wvl) < bestd:
            best, bestd = i, abs(w - target_wvl)
    return best, ids


t = eispac.read_template(eispac.data.get_fit_template_filepath(
    "fe_13_203_826.2c.template.h5"))
print("Fe XIII 203.826 は第", pick_component(t, 203.826)[0], "成分")

## 2-3. 領域を決めて、平均してからフィットする

弱い線は 1 画素では埋もれているので、**まず空間平均して S/N を上げ**、
それからフィットします（速いので Colab でも快適）。

ここでは活動領域コアの中の **inter-moss 領域**（ループの足元ではなく
上部を見ている場所）を使います。

In [ ]:
from workshop import BOX
from fit_box_spectra import average_spectrum      # 欠損値は落としてある（付録 A）

print("箱:", BOX)
wave, inten, sig, npix = average_spectrum(EIS_FILE, 195.119, **BOX)
print(f"平均に使った画素数: {npix}")

fit1 = eispac.fit_spectra(inten, tmplt, wave=wave, errs=sig, ncpu=1,
                          ignore_warnings=True)
wfit, pfit = fit1.get_fit_profile()

plt.figure(figsize=(7, 4.5))
plt.errorbar(wave, inten, yerr=sig, fmt="o", ms=4, label="observed (box average)")
plt.plot(np.ravel(wfit), np.ravel(pfit), "-", lw=2, label="fit")
plt.axvline(195.119, color="r", ls="--", lw=1)
plt.axvline(195.179, color="g", ls="--", lw=1)
plt.xlabel("wavelength [Å]")
plt.ylabel("intensity")
plt.title("Fe XII 195.119")
plt.legend()
plt.tight_layout()
plt.show()

## 2-4. 強度はガウシアンの面積

In [ ]:
p = np.atleast_1d(fit1.fit["params"]).ravel()
A, lam0, sigma = p[0], p[1], p[2]
I_fit = float(np.atleast_1d(fit1.fit["int"][..., 0]).ravel()[0])

print(f"振幅 A     = {A:10.1f}")
print(f"中心 λ0    = {lam0:10.4f} Å")
print(f"幅   σ     = {sigma:10.4f} Å")
print(f"A σ √(2π)  = {A*sigma*np.sqrt(2*np.pi):10.2f}")
print(f"eispac の int = {I_fit:8.2f}   ← 一致する")
print()
print(f"論文 Table 2 の Fe XII 195.119 = 1147.35")
print(f"比 = {I_fit/1147.35:.2f}   （論文の誤差は ±22%）")

**論文の値と 1 割の一致。** 独立に処理した結果が合うのは気持ちがよいところです。

差の主な原因は**測った場所の違い**です。論文は箱の座標を書いていないので、
図から読み取るしかありません。

## 2-5. 演習

1. **別の輝線でやってみる。** テンプレート名は `scripts/lines_warren2012.py` の
   一覧にあります。論文 Table 2 の値と比べてみましょう。
   - `Fe XIII 202.044` → 論文 1076.80
   - `Fe XV 284.160` → 論文 5931.55
   - `Ca XV 200.972` → 論文 127.92
2. **箱を動かす。** `BOX` の y や x をずらすと強度はどれくらい変わるか。
3. `Ca XVII 192.858` をやると論文の 5 倍になります。なぜか考えてみてください
   （ヒント: 第 1 章の Ca XVII のマップは Fe XII に似ていた。答えは付録 G）

In [ ]:
# 演習 1（TODO を埋める）
#
# from lines_warren2012 import LINES
# for ion, wvl, tname, i_paper, sig_paper in LINES:
#     print(f"{ion:8s} {wvl:8.3f}  {tname}")
#
# tmplt2 = eispac.read_template(eispac.data.get_fit_template_filepath("____"))
# wave2, inten2, sig2, _ = average_spectrum(EIS_FILE, ____, **BOX)
# fit2 = eispac.fit_spectra(inten2, tmplt2, wave=wave2, errs=sig2, ncpu=1,
#                           ignore_warnings=True)
# comp2, ids2 = pick_component(tmplt2, ____)
# print(ids2, "→ 第", comp2, "成分")
# print(float(np.atleast_1d(fit2.fit["int"][..., comp2]).ravel()[0]))

## まとめ

- 輝線強度は**ガウシアンの面積** $A\sigma\sqrt{2\pi}$。**フィットの産物**
- **`line_ids` を確認してから成分番号を決める**
- 平均してからフィットすると速く、S/N も上がる
- 論文 Table 2 と 1 割で一致した

# 第 3 章: 速度を出す

**50 分**

同じフィットから、中心波長も出ている。これを**ドップラー速度**に直す。

出てくるのは:

- コロナの物質が**こちらに向かってくるのか、遠ざかるのか**の地図
- 「速度ゼロ」を**自分で決めなければならない**という、分光観測の宿命

Solar-C EUVST は 0.4″・1 秒で同じことをする。**ここで身につけたことは
そのまま使える**（ゼロ点の決め方は装置が変わっても付いて回る）。

## 3-1. ラスターをフィットする

第 2 章では箱の中で平均してから 1 回フィットした。
速度**マップ**がほしいので、今度は**画素ごとにフィット**する。

全部（512×60）だと数分かかるので、活動領域が写っている
y = 180–340 の 160 行だけにする（9600 スペクトル、1〜2 分）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from workshop import fit_region

Y0, Y1 = 180, 340
fit, cube = fit_region(wvl=195.119, y0=Y0, y1=Y1, ncpu=2)   # Fe XII 195.119

cen = fit.fit["params"][..., 1]        # 中心波長 [Å]
sig = fit.fit["params"][..., 2]        # 線幅 σ [Å]  ← 第 4 章で使う
inten = fit.fit["int"][..., 0]         # 強度
print("マップの形:", cen.shape)
print(f"中心波長の範囲: {np.nanmin(cen):.4f} – {np.nanmax(cen):.4f} Å")

## 3-2. 波長のずれを速度に直す

静止波長 $\lambda_0$ からのずれが、視線方向の速度になる:

$$ v = c\,\frac{\lambda_{\rm obs} - \lambda_0}{\lambda_0} $$

Fe XII の静止波長は $\lambda_0 = 195.119$ Å。
1 画素（0.0223 Å）は 195 Å では **34 km/s** に相当する。
つまり**画素の 1/10 以下のずれ**を測ることになる。

In [ ]:
C_KMS = 2.998e5
LAM0 = 195.119

v_naive = C_KMS * (cen - LAM0) / LAM0

print(f"1 波長画素 = {C_KMS*0.0223/LAM0:.0f} km/s")
print(f"素直に計算した速度: 中央値 {np.nanmedian(v_naive):+.1f} km/s  "
      f"5–95% [{np.nanpercentile(v_naive,5):+.1f}, {np.nanpercentile(v_naive,95):+.1f}]")

**視野全体の中央値が 0 になっていない。**

活動領域全体が数 km/s で一様に動いているわけではないので、これは**装置側のずれ**。
理由は次の 2 つで、どちらも分光観測に共通する。

### (a) EIS には絶対的な波長基準が無い

較正用の光源を積んでいないので、「この画素がちょうど 195.119 Å」という
基準が無い。**視野の中の何かをゼロと決めるしかない。**

よく使われる決め方:

| 決め方 | 意味 |
|---|---|
| 視野全体の中央値を 0 | 「平均的には静止している」と仮定する |
| **列（露光）ごとの中央値を 0** | 上に加えて、露光ごとの装置の揺らぎも落とす |
| 静穏領域の値を 0 | 静穏領域が静止していると仮定する |

**どれを選んだかで速度マップの意味が変わる。** 論文には必ず書く。

### (b) 軌道に伴う波長のずれ（eispac が補正済み）

EIS は 98 分で地球を回るあいだに日陰・日照を繰り返し、**装置の温度が変わって
波長が動く**。さらにスリットは検出器に対してわずかに傾いている。

eispac は `read_cube` の時点でこれを補正している（`cube.meta['wave_corr']`）。
**自分で level-0 から処理するなら、この工程は必須。**

In [ ]:
wc = np.asarray(cube.meta["wave_corr"], float)          # (y, x) [Å]
wct = np.asarray(cube.meta["wave_corr_t"], float)       # 時間依存（軌道）
wcx = np.asarray(cube.meta["wave_corr_tilt"], float)    # 位置依存（スリット傾き）

print(f"補正量 全体   : {wc.min():+.4f} 〜 {wc.max():+.4f} Å  "
      f"= {C_KMS*(wc.max()-wc.min())/LAM0:.0f} km/s ぶん")
print(f"  うち 軌道変動: {wct.min():+.4f} 〜 {wct.max():+.4f} Å  "
      f"({C_KMS*(wct.max()-wct.min())/LAM0:.0f} km/s)")
print(f"  うち 傾き    : {wcx.min():+.4f} 〜 {wcx.max():+.4f} Å  "
      f"({C_KMS*(wcx.max()-wcx.min())/LAM0:.0f} km/s)")
print("\n→ コロナの流れ（数十 km/s）と同じ大きさ。補正しなければ速度は測れない。")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(wct, lw=1.5)
axes[0].set_xlabel("raster step (= time)")
axes[0].set_ylabel("wavelength shift [Å]")
axes[0].set_title("orbital drift (time-dependent)")
axes[1].plot(wcx, np.arange(len(wcx)), lw=1.5)
axes[1].set_ylabel("y [pix] (along the slit)")
axes[1].set_xlabel("wavelength shift [Å]")
axes[1].set_title("slit tilt (position-dependent)")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 3-3. ゼロ点を決めて速度マップにする

`eispac.instr.calc_velocity` がやってくれる。
`corr_method` がゼロ点の決め方に対応する。

In [ ]:
from eispac.instr import calc_velocity

v_image = calc_velocity(cen, LAM0, corr_method="image")    # 視野全体の中央値を 0
v_col = calc_velocity(cen, LAM0, corr_method="column")     # 列ごとの中央値を 0

for name, v in [("ゼロ点なし", v_naive), ("視野の中央値を 0", v_image),
                ("列ごとの中央値を 0", v_col)]:
    print(f"{name:20s} 中央値 {np.nanmedian(v):+6.2f}  "
          f"5–95% [{np.nanpercentile(v,5):+6.2f}, {np.nanpercentile(v,95):+6.2f}] km/s")

In [ ]:
ext = [0, 60, Y0, Y1]
fig, axes = plt.subplots(1, 2, figsize=(9, 7))

d = np.sqrt(np.clip(inten, 0, None))
axes[0].imshow(d, origin="lower", aspect="auto", extent=ext, cmap="inferno",
               vmin=0, vmax=np.nanpercentile(d, 99.5))
axes[0].set_title("Fe XII 195.119  intensity")

im = axes[1].imshow(v_col, origin="lower", aspect="auto", extent=ext,
                    cmap="RdBu_r", vmin=-15, vmax=15)
axes[1].set_title("Doppler velocity [km/s]\n(blue = toward us)")
fig.colorbar(im, ax=axes[1], label="km/s")
for ax in axes:
    ax.set_xlabel("x [pix]")
axes[0].set_ylabel("y [pix]")
fig.tight_layout()
plt.show()

強度マップと速度マップは**似ていません**。明るい＝速い、ではない。
明るさと速度の関係を数字で見てみます。

In [ ]:
q20, q80 = np.nanpercentile(inten, [20, 80])
print("強度で 3 分割したときの速度 [km/s]（負 = こちらに向かう = 上昇流）")
for name, m in [("暗い 20%", inten < q20),
                ("中間", (inten >= q20) & (inten < q80)),
                ("明るい 20%", inten >= q80)]:
    print(f"  {name:10s} 中央値 {np.nanmedian(v_col[m]):+5.2f}   "
          f"平均 {np.nanmean(v_col[m]):+5.2f}")

**明るいところはわずかに赤方偏移（下降流）、暗いところはわずかに青方偏移。**
ただし中央値の差は **1–2 km/s** しかありません。

一方、画素ごとの振れ幅は ±15 km/s 程度あります。
つまり**平均的な傾向は小さく、場所ごとのばらつきの方が大きい**。

ここで効いてくるのが 3-2 の話です。**装置由来のずれは 53 km/s ぶん**あり、
測りたい信号より大きい。補正とゼロ点の扱いを間違えれば、
**簡単に嘘の流れが見えます。**

## 3-4. 演習

1. **別の輝線で速度マップを作る。** 温度が違えば速度も違うはず。
   - `Fe XIII 202.044`（1.8 MK、`fe_13_202_044.1c.template.h5`）
   - `Si VII 275.368`（0.6 MK、`si_07_275_368.1c.template.h5`、moss が見える）
   ヒント: `fit_region(wvl=..., tmplt_name=..., y0=Y0, y1=Y1)`

2. **ゼロ点の決め方を変える**（`corr_method="image"` と `"column"`）。
   マップのどこが変わるか。x 方向の縞が出たり消えたりするはず。

3. 速度の**誤差**を見る。`fit.fit["err_params"][..., 1]` が中心波長の誤差。
   暗いところで速度がどれくらい信用できないか確かめる。

In [ ]:
# 演習 1: ____ を埋めて実行してください（Fe XIII 202.044 の速度マップ）
#
# fit2, cube2 = fit_region(wvl=____, tmplt_name="____", y0=Y0, y1=Y1)
# cen2 = fit2.fit["params"][..., ____]
# v2 = calc_velocity(cen2, ____, corr_method="column")
# plt.imshow(v2, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-15, vmax=15)
# plt.colorbar(label="km/s"); plt.show()

**答えは別のノートにあります** →
[演習の答え](https://colab.research.google.com/github/hottahd/EIS_practice/blob/main/notebooks/EIS_workshop_answers.ipynb)

実行結果も入れてあるので、開くだけで確認できます（走らせる必要はありません）。

## まとめ

- 速度は**中心波長のずれ**。1 波長画素が 34 km/s なので、画素の 1/10 以下を測る
- **絶対的な波長基準は無い。ゼロ点は自分で決める**（＝仮定を置く）
- 軌道変動とスリット傾きで **53 km/s ぶん**動く。eispac は補正済み
- **Solar-C でも同じ。** 「速度ゼロを何と決めたか」を言えることが解析の一部

# 第 4 章: 線幅から乱れを測る

**30 分**

フィットから出る 3 つ目の量が**線幅**。ここから
**非熱的速度**（熱運動では説明できない速度成分）が出る。

波・乱流・視線上に重なった細かい流れ —— コロナ加熱の議論に直結する量で、
Solar-C の主戦場のひとつ。

## 4-1. 線幅は 3 つの足し算

観測される線幅は、次の 3 つが**二乗和**で足さったもの:

$$ \sigma_{\rm obs}^2 = \sigma_{\rm inst}^2 + \sigma_{\rm th}^2 + \sigma_{\rm nonth}^2 $$

| | 中身 |
|---|---|
| $\sigma_{\rm inst}$ | **装置の幅**。EIS では最大の寄与 |
| $\sigma_{\rm th}$ | **熱運動**。イオンが重いほど狭い。$\sigma_{\rm th} = \frac{\lambda_0}{c}\sqrt{kT/M}$ |
| $\sigma_{\rm nonth}$ | **それ以外**。波・乱流・視線上の速度の重なり |

非熱的速度 $\xi$ は $\sigma_{\rm nonth} = \frac{\lambda_0}{c}\frac{\xi}{\sqrt{2}}$ で定義する
（$\xi$ は最確速度）。つまり

$$ \xi = \sqrt{2}\,\frac{c}{\lambda_0}
   \sqrt{\sigma_{\rm obs}^2 - \sigma_{\rm inst}^2 - \sigma_{\rm th}^2} $$

**引き算なので、装置幅を間違えると答えが大きく動く。**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from workshop import fit_region

Y0, Y1 = 180, 340
fit, cube = fit_region(wvl=195.119, y0=Y0, y1=Y1, ncpu=2)   # 第 3 章と同じフィット

sig_obs = fit.fit["params"][..., 2]      # フィットで得た σ [Å]
inten = fit.fit["int"][..., 0]
print(f"観測された σ: 中央値 {np.nanmedian(sig_obs):.4f} Å")

## 4-2. 装置幅はスリット上の位置で変わる

EIS の装置幅は `cube.meta['slit_width']` に入っている（**FWHM**、単位 Å）。
**スリットに沿って一定ではない。**

In [ ]:
fwhm_inst = np.asarray(cube.meta["slit_width"], float)[Y0:Y1]
sig_inst = fwhm_inst / (2 * np.sqrt(2 * np.log(2)))      # FWHM → σ

print(f"装置幅 FWHM: {fwhm_inst.min():.4f} – {fwhm_inst.max():.4f} Å "
      f"（この範囲で {100*(fwhm_inst.max()/fwhm_inst.min()-1):.0f}% 変わる）")
print(f"σ に直すと : {sig_inst.min():.4f} – {sig_inst.max():.4f} Å")
print(f"観測の σ   : 中央値 {np.nanmedian(sig_obs):.4f} Å")
print("\n→ 観測した幅のほとんどが装置の幅。差の部分を取り出す作業になる。")

plt.figure(figsize=(4.5, 5))
plt.plot(fwhm_inst, np.arange(Y0, Y1), lw=1.5)
plt.xlabel("instrumental width FWHM [Å]")
plt.ylabel("y [pix] (along the slit)")
plt.title("instrumental width varies along the slit")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4-3. 熱幅を引く

熱運動の幅は**イオンの質量と温度**で決まる。
温度は測れないので、**その輝線の形成温度を仮定する**（Fe XII なら log T = 6.2）。

これは仮定なので、後で変えてみて影響を見る（演習 2）。

In [ ]:
LAM0, C_KMS = 195.119, 2.998e5
M_AMU, LOGT = 56.0, 6.2                  # Fe XII: 鉄の質量数と形成温度
K_B, AMU = 1.380649e-16, 1.66054e-24     # cgs

v_th = np.sqrt(K_B * 10**LOGT / (M_AMU * AMU)) / 1e5      # km/s
sig_th = LAM0 * v_th / C_KMS

print(f"熱速度   sqrt(kT/M) = {v_th:.1f} km/s   （logT={LOGT}, 鉄）")
print(f"熱幅     σ_th       = {sig_th:.4f} Å")
print(f"装置幅   σ_inst     = {np.median(sig_inst):.4f} Å  ← こちらの方がずっと大きい")

## 4-4. 非熱的速度のマップ

In [ ]:
def nonthermal_velocity(sig_obs, sig_inst, sig_th, lam0=LAM0):
    """観測の σ から非熱的速度 ξ [km/s] を出す。引けない画素は NaN。"""
    excess = sig_obs**2 - sig_inst**2 - sig_th**2
    excess = np.where(excess > 0, excess, np.nan)
    return np.sqrt(2) * C_KMS / lam0 * np.sqrt(excess)


xi = nonthermal_velocity(sig_obs, sig_inst[:, None], sig_th)   # ← y ごとの装置幅

bright = inten > np.nanpercentile(inten, 60)      # 暗い場所は幅が信用できない
print(f"非熱的速度（明るい画素）: 中央値 {np.nanmedian(xi[bright]):.1f} km/s   "
      f"5–95% [{np.nanpercentile(xi[bright],5):.1f}, {np.nanpercentile(xi[bright],95):.1f}]")
print(f"引き算が成立しなかった画素: {100*np.isnan(xi[bright]).mean():.0f}%")

In [ ]:
ext = [0, 60, Y0, Y1]
fig, axes = plt.subplots(1, 2, figsize=(9, 7))
d = np.sqrt(np.clip(inten, 0, None))
axes[0].imshow(d, origin="lower", aspect="auto", extent=ext, cmap="inferno",
               vmin=0, vmax=np.nanpercentile(d, 99.5))
axes[0].set_title("Fe XII 195.119  intensity")

im = axes[1].imshow(np.where(bright, xi, np.nan), origin="lower", aspect="auto",
                    extent=ext, cmap="viridis", vmin=5, vmax=35)
axes[1].set_title("non-thermal velocity [km/s]")
fig.colorbar(im, ax=axes[1], label="km/s")
for ax in axes:
    ax.set_xlabel("x [pix]")
axes[0].set_ylabel("y [pix]")
fig.tight_layout()
plt.show()

明るさとの関係を数字で見ます。

In [ ]:
q20, q80 = np.nanpercentile(inten, [20, 80])
print("強度で 3 分割したときの非熱的速度 [km/s]")
for name, m in [("暗い 20%", inten < q20),
                ("中間", (inten >= q20) & (inten < q80)),
                ("明るい 20%", inten >= q80)]:
    print(f"  {name:10s} 中央値 {np.nanmedian(xi[m]):5.1f}")
good = np.isfinite(xi) & (inten > 0)
print(f"\n相関係数 (log I, ξ) = {np.corrcoef(np.log10(inten[good]), xi[good])[0,1]:+.2f}")

- 全体の中央値は **18 km/s 前後**。活動領域として典型的な値
- **明るいところほど非熱的速度は大きい**（暗 14 → 明 20 km/s）。
  ただし相関は +0.1 程度で**弱い**
- 暗い画素は線幅の測定誤差が大きく、引き算が破綻して NaN になる
  （上の統計で暗い側の値が低めに出るのは、この効果も混ざっている）

「非熱的」の中身は 1 つではない。波かもしれないし、視線上に速度の違う
構造がいくつも重なっているだけかもしれない。
**0.4″ の Solar-C で見ると、この一部は「分解できていなかっただけ」に変わるはず。**
それを確かめるのが EUVST の仕事のひとつ。

## 4-5. 演習

1. **装置幅を定数（平均値）で代用するとどうなるか。**
   `nonthermal_velocity(sig_obs, sig_inst.mean(), sig_th)` に変えて引き算し、
   差のマップを描く。**y 方向に系統的なパターン**が出るはず。
2. **形成温度の仮定を変える。** `LOGT` を 6.0 や 6.4 にすると ξ はどれだけ動くか。
   「仮定が結果に効く」量なのかどうかを自分で確かめる。
3. 軽い元素の輝線（例: `Si X 258.375`、Si は鉄より軽い）で同じことをする。
   熱幅の寄与が大きくなるので、仮定の効き方も変わる。

In [ ]:
# 演習 1: ____ を埋めて実行してください（装置幅を定数で代用するとどうなるか）
#
# xi_const = nonthermal_velocity(sig_obs, ____, sig_th)
# diff = xi - xi_const
# plt.imshow(np.where(bright, diff, np.nan), origin="lower", aspect="auto",
#            extent=ext, cmap="coolwarm", vmin=-10, vmax=10)
# plt.colorbar(label="km/s"); plt.show()

**答えは別のノートにあります** →
[演習の答え](https://colab.research.google.com/github/hottahd/EIS_practice/blob/main/notebooks/EIS_workshop_answers.ipynb)

実行結果も入れてあるので、開くだけで確認できます（走らせる必要はありません）。

## まとめ

- 線幅は **装置 + 熱運動 + それ以外** の二乗和。**引き算で非熱的速度を取り出す**
- **装置幅はスリット上の位置で変わる**（この観測で 7%）。`slit_width` を使う
- 熱幅には**温度の仮定**が入る
- 活動領域の非熱的速度は **20 km/s 前後**

# 第 5 章: 温度分布を出す

**30 分**

第 1 章で見た「輝線ごとにまったく違う絵」の正体を、数字にします。

視線上には色々な温度のプラズマが混ざっています。
**どの温度がどれだけあるか**を求めるのが **DEM（emission measure 分布）解析**です。

## 5-1. 輝線は温度計になる

光学的に薄いコロナでは、輝線強度は視線上の足し算になります:

$$ I_\lambda = \frac{1}{4\pi}\int G_\lambda(T)\, n_e n_H\, ds $$

$G_\lambda(T)$ が**寄与関数**で、「その輝線がどの温度で光るか」を表します。
中身は **組成 × 電離平衡 × 励起**で、原子データベース（CHIANTI）から計算します。

**温度の幅が狭いのは電離平衡のおかげ**です。各イオンは log T で 0.2–0.3 dex
の範囲でしか存在できません（低温では電離しておらず、高温ではさらに電離する）。

CHIANTI の計算には時間がかかるので、**事前に計算したものを同梱**しています。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from workshop import box_intensities


def read_gofnt(path):
    """事前計算した G(T) を読む。"""
    lines = open(path).readlines()
    i = next(k for k, l in enumerate(lines) if l.startswith("# nT nline"))
    nT, nline = (int(v) for v in lines[i + 1].split())
    k = i + 2

    def skip(tag):
        nonlocal k
        while not lines[k].startswith(tag):
            k += 1
        k += 1

    def take(n):
        nonlocal k
        out = []
        while len(out) < n:
            out += [float(x) for x in lines[k].split()]
            k += 1
        return np.array(out[:n])

    skip("# logT")
    logT = take(nT)
    skip("# ion")
    names, wvl = [], []
    for _ in range(nline):
        p = lines[k].split()
        names.append(" ".join(p[:-2]))
        wvl.append(float(p[-2]))
        k += 1
    skip("# G(T)")
    G = np.array([take(nT) for _ in range(nline)])
    return logT, names, np.array(wvl), G


logT, names, wvl, G = read_gofnt("work/gofnt_chianti901_005.txt")
print(f"{len(names)} 輝線 × {len(logT)} 温度点  "
      f"(logT {logT[0]:.1f}–{logT[-1]:.1f}, {logT[1]-logT[0]:.2f} dex 刻み)")

fig, ax = plt.subplots(figsize=(8, 4.5))
cmap = plt.get_cmap("turbo")
tpk = np.array([logT[int(np.argmax(g))] for g in G])
norm = plt.Normalize(tpk.min(), tpk.max())
for k in range(len(names)):
    ax.plot(logT, G[k] / G[k].max(), color=cmap(norm(tpk[k])), lw=1.2)
ax.set_xlim(5.4, 7.2)
ax.set_xlabel("log T [K]")
ax.set_ylabel("G(T) / max")
ax.set_title("contribution functions (color = peak temperature)")
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, label="log T at peak")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

22 本の輝線が log T = 5.8 から 6.9 まで**少しずつずれた温度**に効いています。
これが温度分布を測る武器です。

ただし**曲線は幅広く重なっています**。だから 22 本測っても
22 個ぶんの独立な情報にはなりません（後述）。

## 5-2. EM loci —— 解く前に答えの見当をつける

輝線ごとに「**もし視線上のプラズマが全部ちょうど温度 T にあったら、
EM はいくら必要か**」を計算します:

$$ {\rm EM}_{\rm loci}(T) = \frac{4\pi I_\lambda}{G_\lambda(T)} $$

これは各温度における **EM の上限**で、真の値は必ずこの下にあります。
**等温プラズマなら全部の曲線が 1 点で交わります。**

In [ ]:
rows = box_intensities()        # 第 2 章と同じ箱の 22 輝線（無ければその場で作る）

iobs = np.zeros(len(names))
for ion, w, i_fit, i_paper, ratio in rows:
    k = int(np.argmin(np.abs(wvl - w)))
    if abs(wvl[k] - w) < 0.01:
        iobs[k] = i_fit
iobs[int(np.argmin(np.abs(wvl - 192.858)))] = 0.0     # Ca XVII はブレンド（付録 G）

ok = np.where(iobs > 0)[0]
fig, ax = plt.subplots(figsize=(8.5, 5.5))
tpk_ok = np.array([logT[int(np.argmax(G[k]))] for k in ok])
norm = plt.Normalize(tpk_ok.min(), tpk_ok.max())
env = np.full(len(logT), np.inf)
for k, tp in zip(ok, tpk_ok):
    g = G[k]
    m = g > g.max() * 1e-3
    loci = 4 * np.pi * iobs[k] / np.where(g > 0, g, np.nan)
    ax.plot(logT[m], loci[m], color=cmap(norm(tp)), lw=1.3, alpha=0.9)
    j = int(np.nanargmin(np.where(m, loci, np.inf)))
    ax.annotate(f"{names[k]} {wvl[k]:.1f}", (logT[j], loci[j]), fontsize=6.5,
                color=cmap(norm(tp)), xytext=(2, 2), textcoords="offset points")
    env = np.minimum(env, np.where(m, loci, np.inf))
ax.plot(logT, env, "k--", lw=1.8, label="lower envelope = upper limit")
ax.set_yscale("log")
ax.set_xlim(5.4, 7.2)
ax.set_ylim(1e25, 1e31)
ax.set_xlabel("log T [K]")
ax.set_ylabel(r"EM$_{\rm loci} = 4\pi I / G(T)$  [cm$^{-5}$]")
ax.set_title("EM loci: EM required if ALL the plasma were at temperature T")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, label="log T at peak")
fig.tight_layout()
plt.show()

**1 点で交わっていません。→ 等温ではない。多温度です。**

- 低温側（Si VII, Fe IX）の曲線が高いところにある = **冷たいプラズマは少ない**
- 曲線の底が log T 6.1–6.6 に集まっている = **本体はこの温度域**
- log T > 6.9 では拘束する輝線が無く、上限が跳ね上がる
  → 本当は AIA 94 Å の Fe XVIII（7 MK）を足したいところ（付録 F）

## 5-3. DEM を解く

$I_\lambda = \frac{1}{4\pi}\int G_\lambda(T)\,\xi(T)\,dT$ を $\xi(T)$ について解きます。
ここでは正則化インバージョン（`demregpy`）を使います。

誤差は論文と同じ **22%**（統計誤差 + 較正の不確かさ）を使います（付録 B）。

In [ ]:
from demregpy import dn2dem

keep = (logT >= 5.5) & (logT <= 7.1)
lt, Gk = logT[keep], G[:, keep]
dlt = lt[1] - lt[0]

sel = iobs > 0
dn = iobs[sel]
edn = 0.22 * dn                       # 較正の系統誤差（付録 B）
tresp = (Gk[sel] / (4 * np.pi)).T     # I = (1/4π)∫G ξ dT なので G/(4π)
tedges = 10 ** np.append(lt - dlt / 2, lt[-1] + dlt / 2)

dem, edem, elogt, chisq, dn_reg = dn2dem(dn, edn, tresp, lt, tedges,
                                         max_iter=30, warn=False)
dem = np.atleast_1d(np.squeeze(dem))
T = 10**lt
em = dem * T * np.log(10) * dlt       # DEM [cm^-5/K] → ビンあたりの EM [cm^-5]

ipk = int(np.nanargmax(em))
print(f"使った輝線     : {int(sel.sum())} 本")
print(f"reduced chi2   : {float(np.squeeze(chisq)):.2f}")
print(f"EM のピーク    : logT {lt[ipk]:.2f}  = {T[ipk]/1e6:.1f} MK")
print(f"総 EM          : {em.sum():.2e} cm^-5")
print(f"  → n_e = 1e9 cm^-3 なら視線長 {em.sum()/1e18/1e8:.0f} Mm（妥当なオーダー）")

In [ ]:
plt.figure(figsize=(7.5, 5))
plt.plot(lt, em, "o-", lw=2, ms=4)
plt.axvline(lt[ipk], color="r", ls="--", lw=1,
            label=f"peak: {T[ipk]/1e6:.1f} MK")
plt.yscale("log")
plt.xlim(5.6, 7.1)
plt.ylim(1e24, 1e28)
plt.xlabel("log T [K]")
plt.ylabel(r"EM per bin [cm$^{-5}$]")
plt.title("emission measure distribution (inter-moss region, NOAA 1243)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5-4. 何が分かったか

**活動領域コアの emission measure は 4 MK 付近に強くピークを持つ。**
論文（Warren+2012）の主要な結論と同じです。

これが効いてくるのは**コロナ加熱の議論**です。

- 加熱が**まれ**（ナノフレア的）なら、ループは加熱のあいだに冷える時間があり、
  色々な温度のプラズマが視線上に溜まる → **EM 分布は幅広くなる**
- 加熱が**頻繁**なら、ループは冷える暇がなく高温に保たれる
  → **EM 分布は鋭くピークを持つ**

観測されたピークの鋭さは、**加熱が高頻度**であることを示唆します。
Parker のナノフレア描像（低頻度加熱）への挑戦になっている、というのが論文の主張です。

**ただし、この「鋭さ」は解き方に依存します。** ピーク温度は動きませんが、
傾きは手法や設定で変わります（付録 C）。
DEM は逆問題で、**必ず何らかの仮定が入る**ことは覚えておいてください。

## 5-5. 演習

1. **輝線を減らして解いてみる。** 高温側（Ca XIV–XVI）を落とすと
   ピークはどうなるか。`sel` を書き換えて確かめる
2. **誤差を 22% から 10% に変える。** chi2 と解の形はどう変わるか
3. EM loci の**包絡線の下に**、解いた EM がちゃんと収まっているか確認する
   （超えていたらそれだけで間違い）

In [ ]:
# 演習 1: ____ を埋めて実行してください（高温側の輝線を落とすとどうなるか）
#
# sel2 = sel.copy()
# for bad in ["Ca XIV", "Ca XV", "Ca XVI"]:
#     sel2[[i for i, n in enumerate(names) if n == bad]] = ____
# dem2, _, _, chi2_2, _ = dn2dem(iobs[sel2], 0.22*iobs[sel2],
#                                (Gk[sel2]/(4*np.pi)).T, lt, tedges,
#                                max_iter=30, warn=False)
# em2 = np.atleast_1d(np.squeeze(dem2)) * T * np.log(10) * dlt
# plt.plot(lt, em, "o-", label="all lines")
# plt.plot(lt, em2, "s-", label="without Ca")
# plt.yscale("log"); plt.legend(); plt.show()

**答えは別のノートにあります** →
[演習の答え](https://colab.research.google.com/github/hottahd/EIS_practice/blob/main/notebooks/EIS_workshop_answers.ipynb)

実行結果も入れてあるので、開くだけで確認できます（走らせる必要はありません）。

## 今日のまとめ

| 章 | 出したもの | Solar-C で |
|---|---|---|
| 1 | 輝線ごとの強度マップ | 同じ輝線を 0.4″ で |
| 2 | **輝線強度**（ガウシアンの面積） | そのまま使う |
| 3 | **ドップラー速度**（ゼロ点は自分で決める） | そのまま使う |
| 4 | **非熱的速度**（装置幅を引く） | そのまま使う |
| 5 | **温度分布**（4 MK にピーク） | 温度被覆が広がってもっと良く決まる |

**今日の手順は、装置が変わってもそのまま使えます。**
2028 年に EUVST のデータが降りてきたら、同じことをすればいい。

付録には、今日触れなかった話（欠損値、誤差、DEM の信頼度、較正、
ブレンド分離など）をまとめてあります。自分の研究で使うときに読んでください。

# 付録

**講習会では走らせません。** 自分の研究で EIS / EUVST を使うときに読んでください。

本編で使っているコードは、ここに書いてあることを**既に正しく処理**しています。
「なぜそうしているのか」を知りたくなったときの参照先です。

| | 内容 |
|---|---|
| A | 欠損値は NaN ではない |
| B | 誤差の入れ方 |
| C | DEM をどこまで信じてよいか |
| D | G(T) と 1/(4π) の罠 |
| E | 較正 |
| F | AIA 94 Å から Fe XVIII を分離する |
| G | Ca XVII のブレンド分離 |
| H | 論文 Table 2 との全面照合 |
| I | 他の活動領域・15 領域の統計 |

## 付録 A: 欠損値は NaN ではない

**EIS の level-1 では、不良画素・宇宙線で捨てられたサンプルは
大きな負のフラグ値**（level-0 の `-100` に較正係数を掛けたもの。
実際には −1000 〜 −6000 程度）として入っています。NaN ではありません。

したがって:

- `np.isnan` / `np.nansum` では**素通りします**
- 足すと大きな負の数が入り、その行だけ暗くなります（強度マップに**横縞**が出る）
- **エラーは一切出ません**

eispac は `cube.mask`（True = 使ってはいけない）を立ててくれるので、これで落とします:

```python
d = np.where(np.asarray(cube.mask, dtype=bool), np.nan, cube.data)
img = np.nanmean(d, axis=2) * d.shape[2]
```

論文 §3 も *"In computing these averaged profiles, missing data are not included"*
とわざわざ書いています。

**影響の実測**（本編の箱、22 輝線）: median は 0%、大半の線で 1% 未満。
ただし弱い線では効きます（Ca XVII +6.9%、Ca XVI −4.6%、Ar XIV +3.2%）。
IDL 側では最も明るい Fe XII 195.119 が 24% 小さくなった例もあります。

**EUVST でも同じ発想が要ります。** 「欠損をどう表現しているか」は
データ形式ごとに違うので、**必ず確認してから平均を取る**こと。

## 付録 B: 誤差の入れ方

箱の中で数百画素を平均すると、**統計誤差は 0.2% 程度**まで落ちます。
一方、論文が使っている誤差は **22%** です。

差は**絶対較正の不確かさ**（系統誤差）で、これは平均しても減りません。

統計誤差だけで DEM を解こうとすると、モデル（滑らかな DEM）の
わずかな不完全さが全部 χ² に化けて**発散**します。

```python
edn = np.maximum(stat_err, 0.22 * intensity)   # 較正誤差を「床」として入れる
```

**「誤差が小さい」ことは良いことではありません。**
何の誤差を見積もっているのかを意識してください。

## 付録 C: DEM をどこまで信じてよいか

DEM は**第一種 Fredholm 積分方程式**の逆問題で、本質的に ill-posed です。

**実質的な自由度は輝線の本数より少ない。**
22 輝線 × 33 温度ビンの応答行列を特異値分解すると、
最大の 1/1000 以上ある特異値は **12 本**しかありません
（G(T) が幅広く重なっているため）。

```python
sv = np.linalg.svd(G / (4*np.pi), compute_uv=False)
(sv > 1e-3 * sv[0]).sum()      # → 12
```

**手法・設定で答えがどれだけ動くか**（同じ強度・同じ G(T) で実測）:

| 設定 | chi2 | EM ピーク | 傾き α (logT 6.0–6.6) |
|---|---:|---|---:|
| demregpy 既定 | 3.43 | 6.65 (4.5 MK) | 1.86 |
| demregpy reg_tweak=2 | 4.48 | 6.65 | 1.55 |
| demregpy（MCMC を事前分布に） | 1.49 | 6.60 (4.0 MK) | 2.22 |
| MCMC_DEM (PINTofALE) | — | 6.60 (4.0 MK) | 2.30 |
| 論文 Table 1 | — | ~4 MK | 2.9 |

→ **ピーク温度 4 MK はどの手法でも動かない**（頑健な結論）。
**傾き α は 1.5–2.3 と動く**（慎重に扱うべき量）。
加熱の頻度を α で議論するなら、この系統誤差を押さえる必要があります。

**単位の罠**: DEM の単位は実装で違います。

| | DEM の単位 | ビンあたりの EM |
|---|---|---|
| PINTofALE | cm⁻⁵ **/ logK** | DEM × ΔlogT |
| demregpy | cm⁻⁵ **/ K** | DEM × T ln10 ΔlogT |

揃えないと 10⁷ ずれます。**傾きも +1 ずれます**（実際に間違えました）。

## 付録 D: G(T) と 1/(4π) の罠

同じ CHIANTI 9.0.1 のファイルを 3 つの実装に読ませて G(T) を比べた結果:

| 実装 | 単位の約束 | CHIANTI IDL との比 |
|---|---|---|
| CHIANTI IDL `emiss_calc` | 4π で割らない | 1.000 |
| fiasco `contribution_function` | 4π で割らない | 0.972 – 1.035 |
| **ChiantiPy `ion.emiss()`** | **sr⁻¹（4π で割ってある）** | **0.077 – 0.082** |

**1/(4π) = 0.0796。12.6 倍ずれます。しかもエラーは出ません。**
全輝線が一律にずれるので、線ごとの比を見ている限り気づけません。

**気づく方法はオーダーの検算だけ**です:

| 量 | 覚えておく値 |
|---|---|
| Fe XII 195.119 の G ピーク | ~1.4×10⁻²³ erg cm³ s⁻¹ |
| 活動領域の EM（視線積分） | 10²⁷ – 10²⁹ cm⁻⁵ |
| コロナの電子密度 | ~10⁹ cm⁻³ |
| → 視線長 L = EM/n_e² | ~100 Mm（妥当） |

残る 3% の差は**準安定準位の占有数**の扱いによるもので、
**密度敏感線（Fe XIII 202/204 など）に集中**しています。
Fe XIII が DEM で最も外れる線であることの独立な傍証になっています。

自分で計算するなら **fiasco** が扱いやすい（CHIANTI IDL と 3% 以内で一致）。
実装は `scripts/gofnt_fiasco.py`。

## 付録 E: 較正

EIS は 2006 年打ち上げで、**有機物の付着などで感度が落ちています**。
しかも**波長によって落ち方が違います**。

打ち上げ後較正が 2 つ提案されていますが（Del Zanna 2013、Warren et al. 2014）、
**両者は食い違います**。どちらを使ったかで強度が数十 % 変わります。

→ **論文には必ず「どの較正を使ったか」を書く。**
他人の値と比べるときは、まず較正を揃える。

実装は `scripts/idl/11_calcurve.pro`、結果は `work/eis_calcurve_20110702.txt`。

**絶対較正は 20% 程度ずれるもの**、という感覚を持っておくとよいです。
EUVST でも同じ問題は必ず起きます。

## 付録 F: AIA 94 Å から Fe XVIII を分離する

EIS で観測できる最高温の強い輝線は Ca XVII 192.858（~5 MK）です。
**それより上を拘束するものが無い**と、EM 分布の高温側が決まりません
（第 5 章の EM loci で、log T > 6.9 の上限が跳ね上がっていたのがこれ）。

そこで **AIA 94 Å の Fe XVIII 93.932 Å（~7 MK）**を使います。
ただし 94 Å チャンネルは低温の線に汚染されているので、
171 Å と 193 Å から「低温成分」を経験的に見積もって引きます（論文 Appendix）。

```
x        = (0.31 I_171 + 0.69 I_193) / 116.54
I_94warm = 0.39 (a1 + a2 x + a3 x^2 + a4 x^3)
a        = [-7.31e-2, 9.75e-1, 9.90e-2, -2.84e-3]
I_FeXVIII = I_94 - I_94warm
```

**★ 論文に印刷されている式の指数は誤植です。** 字面どおり
$\sum a_i x^i$ と読むと warm 成分が観測値を桁違いに超えます。
正しくは**定数項つきの 3 次式**（実データで確認）。

適用限界: フレア中は不可（Fe XXIV が 193 Å に入る）、
明るい moss でも破綻、AIA の感度劣化補正を掛けてはいけない。

DEM に入れるときは、**公式の 94 Å 応答は使えません**（低温線込みのため）。
Fe XVIII だけの応答関数を作ってあります（`work/aia94_fe18_response.txt`、
ピーク 2.73×10⁻²⁷ DN cm⁵ s⁻¹ pix⁻¹ at log T 6.90）。

実装は `scripts/aia_fe18.py`, `scripts/aia94_fe18_response.py`。

## 付録 G: Ca XVII のブレンド分離

第 2 章の演習 3 で Ca XVII 192.858 が論文の 5 倍になったのは、
**Fe XI 192.813 と O V の複合線に埋もれている**ためです。
eispac 同梱の `ca_17_192_858.1c` は、この波長域を単一ガウシアンで塗るだけで
ブレンドを分離しません。

論文は Ko et al. (2009) の方法で分離しています。
**自作テンプレート**で同じことができます（`scripts/ca17_template.py`）:

| | Ca XVII 192.858 | 論文比 |
|---|---:|---:|
| eispac 同梱（1 成分） | 198.5 | 4.75 |
| **自作 5 成分テンプレート** | **31.3** | **0.75** |
| SSW/IDL 版 | 32.0 | 0.77 |

設計の要点:

- **分離できない成分は統合する。** EIS のサンプリング 0.0223 Å に対し、
  O V の 192.797/192.801 は分離不能 → 強度重み付き波長で 1 本にまとめる
- 自由パラメータは 5 つだけ。他は**原子データと他の輝線から固定**
  （O V の分岐比、Fe XI 192.813 = Fe XI 188.216 × 0.20896、
    Ca XVII の線幅 = Ca XIV 193.874 の線幅）

**成分を増やせば良くなるわけではない**、というのがこの作業の教訓です。

## 付録 H: 論文 Table 2 との全面照合

22 輝線すべてを論文と比べると、**median 0.89、21 本中 13 本が 15% 以内**
（箱 y=[244:274] x=[32:40]）。論文の誤差 ±22% の中に収まります。

**★ 箱の選び方の診断法**: ratio を**形成温度に対して**並べ、傾きを見ます。

| 傾き | 意味 |
|---|---|
| ≈ 0 | 論文と同じ温度組成の場所を見ている |
| < 0 | 暖かいループ寄りを選んでいる |
| > 0 | 高温コア寄りを選んでいる |

較正は波長の関数であって温度の関数ではないので、
**温度に沿ったパターンが出たら、それは場所の違い**です。

**★ 要約統計 1 つで判断しない**（実測）:

| 箱 | median | 傾き | 15% 以内 | ばらつき |
|---|---:|---:|---:|---:|
| inter-moss（採用） | 0.89 | +0.21 | 13/21 | 0.10 dex |
| 論文の箱サイズ | 0.93 | +0.27 | 15/21 | 0.12 dex |
| **適当に明るいところ** | **0.96** | **−0.34** | **5/21** | **0.26 dex** |

一番下は **median が最も 1 に近いのに、15% 以内は 5 本しかありません**。
個々の線が上下に外れて打ち消し合っているだけです。

実装は `scripts/compare_table2.py`。

## 付録 I: 他の活動領域・15 領域の統計

論文 Table 1 には 15 の活動領域があります（`docs/01_paper_analysis.md` に全リスト）。
`scripts/fetch_data.py --region N` で別の領域のデータを取れます。

- region 8 (2010-07-23, NOAA 1089) は Warren et al. 2011 と同じ活動領域
- region 1 (2010-06-19) は **Ca XIV–XVI が無い**
  → 「観測プログラムによって使える輝線が違う」実例。3 MK 以上を拘束できない

論文の主要な図:

- Figure 4: 高温放射 I_hot と磁束 Φ_M の関係（べき指数 2.3）
- Figure 9: 暖かい成分の EM と Φ_M は**逆相関**

自分が興味を持っている活動領域で同じ解析をしてみるのが、次の一歩です。

---

## この教材について

- リポジトリ: https://github.com/hottahd/EIS_practice
- ライセンス: CC BY 4.0（出典を示せば自由に使えます）
- 検証済みの解析スクリプトは `scripts/`、準備の記録は `docs/`

元になった論文:
Warren, H. P., Winebarger, A. R., & Brooks, D. H. 2012,
*"A Systematic Survey of High-Temperature Emission in Solar Active Regions"*,
ApJ, 759, 141 — https://doi.org/10.1088/0004-637X/759/2/141